# Part 3.1 -- UNet (ResNet-34)

**Part 3 literature comparison (Centralized)** -- protocol-identical to the Part-2 FSCG-UNet run:
same 320px input, same Part-1 sample JSONs + pre-resize cache, same oversampled 2560 train set
and two-pipeline augmentation, same composite loss (CE+Dice+Focal+Lovasz), same AdamW /
CosineAnnealingWarmRestarts / AMP / grad-clip / 8h budget, and final evaluation on the **same
held-out REAL test split with TTA**. Only the model architecture differs from FSCG-UNet.

Centralized training of **UNet (ResNet-34)** under the exact Part-2 recipe.

**Reference:** [36] Karcı et al., *Natural Hazards*, 2026 -- UNet+ResNet comparison

---
### What you need attached on Kaggle
1. The FloodNet dataset (`aerial-imagery-dataset-floodnet-challenge` + `floodnet_fixed`).
2. The **feds1 model** (Part 1 + Part 2 outputs) as an input -- auto-discovered at
   `/kaggle/input/models/YOUR_KAGGLE_USERNAME/fedall/...`. If missing, nothing will run.

## ⚙️ Install packages

In [ ]:
import subprocess, sys
packages = [
    "segmentation-models-pytorch==0.3.3",
    "albumentations==1.3.1",
    "timm",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Packages ready.")

## 📚 Imports

In [ ]:
import os, sys, json, random, warnings, time, pickle, copy, math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler

warnings.filterwarnings("ignore")
print("Imports OK. PyTorch", torch.__version__)

In [ ]:
# ===== Local override (RunPod real full run) =====
SMOKE_MODE = False   # real full-scale run

os.environ["PART1_DIR"]     = "/workspace/fscg/part1_outputs/fed_salgate_outputs"
os.environ["FLOODNET_BASE"] = "/workspace/data/floodnet/FloodNet Challenge - Track 1"
os.environ["SECONDARY_BASE"] = "/workspace/data/floodnet_fixed/data_finally"
os.environ["FSCG_SMOKE"]    = "1" if SMOKE_MODE else "0"
os.environ["FSCG_WORKERS"]  = "0" if SMOKE_MODE else "8"

print("Local override active - SMOKE_MODE =", SMOKE_MODE)
print({k: os.environ[k] for k in ["PART1_DIR", "FLOODNET_BASE", "SECONDARY_BASE", "FSCG_SMOKE", "FSCG_WORKERS"]})

## ⚙️ Config + Part-1 discovery

In [ ]:
SMOKE_TEST  = os.environ.get("FSCG_SMOKE") == "1"
DEBUG_SHORT = os.environ.get("FSCG_DEBUG") == "1" or SMOKE_TEST

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IN_KAGGLE = os.path.isdir("/kaggle/input")
if IN_KAGGLE:
    WORKING_DIR = "/kaggle/working"
else:
    WORKING_DIR = os.path.join(os.path.dirname(os.getcwd()), "working")
    os.makedirs(WORKING_DIR, exist_ok=True)

OUTPUT_DIR = os.path.join(WORKING_DIR, "fed_salgate_outputs")
CKPT_DIR   = os.path.join(WORKING_DIR, "checkpoints")
PLOT_DIR   = os.path.join(OUTPUT_DIR, "plots")
for d in [OUTPUT_DIR, CKPT_DIR, PLOT_DIR, os.path.join(OUTPUT_DIR, "tables")]:
    os.makedirs(d, exist_ok=True)

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
if HAS_CUDA:
    torch.backends.cudnn.benchmark = True
print("Device:", DEVICE)


def _find_part1_dir():
    if os.path.exists(os.path.join(OUTPUT_DIR, "config.json")):
        return OUTPUT_DIR
    _pin = "/kaggle/input/models/YOUR_KAGGLE_USERNAME/fedall/tensorflow2/default/1/fed_salgate_outputs"
    if IN_KAGGLE and os.path.exists(os.path.join(_pin, "config.json")):
        return _pin
    if IN_KAGGLE:
        for root, dirs, files in os.walk("/kaggle/input"):
            if "config.json" in files and "client_indices.pkl" in files:
                if os.path.basename(root) == "fed_salgate_outputs":
                    return root
    return None


PART1_DIR = os.environ.get("PART1_DIR", "") or (_find_part1_dir() or "")
print(f"Part 1 outputs: {PART1_DIR}")
if not PART1_DIR or not os.path.exists(os.path.join(PART1_DIR, "config.json")):
    raise FileNotFoundError("Part-1 outputs not found. Attach the feds1 model (Part 1 output).")

with open(os.path.join(PART1_DIR, "config.json")) as f:
    CFG = json.load(f)

NUM_CLASSES = int(CFG.get("NUM_CLASSES", 10))
CLASS_NAMES = CFG.get("CLASS_NAMES", ['Background', 'Building-Flooded', 'Building-NonFlooded',
                                      'Road-Flooded', 'Road-NonFlooded', 'Water',
                                      'Tree', 'Vehicle', 'Pool', 'Grass'])
IMG_SIZE   = int(os.environ.get("FSCG_IMG", "320"))
BATCH_SIZE = int(os.environ.get("FSCG_BATCH", "4")) if not SMOKE_TEST else 2
class_weights = CFG.get("class_weights")
NUM_CLIENTS = int(CFG.get("NUM_CLIENTS", 3))
print(f"IMG_SIZE={IMG_SIZE}  BATCH={BATCH_SIZE}  NUM_CLASSES={NUM_CLASSES}")

FSCG_EPOCHS    = os.environ.get("FSCG_EPOCHS", "3" if SMOKE_TEST else "100")
FSCG_MAX_HOURS = os.environ.get("FSCG_MAX_HOURS", "8.0")
FSCG_VALLOG    = os.environ.get("FSCG_VALLOG", "64")
FSCG_WORKERS   = os.environ.get("FSCG_WORKERS", "4")
EPOCHS    = int(FSCG_EPOCHS)
MAX_HOURS = float(FSCG_MAX_HOURS)
VAL_LOG_N = int(FSCG_VALLOG)
WORKERS   = int(FSCG_WORKERS)
print(f"EPOCHS={EPOCHS} MAX_HOURS={MAX_HOURS}h VAL_LOG_N={VAL_LOG_N} WORKERS={WORKERS}")
print(">>> Literature baseline: SAME data/aug/loss/optimizer/budget as Part-2 FSCG-UNet (320px). "
      "Only the architecture differs.")

_client_pkl = os.path.join(PART1_DIR, "client_indices.pkl")
CLIENT_INDICES = None
if os.path.exists(_client_pkl):
    with open(_client_pkl, "rb") as f:
        CLIENT_INDICES = pickle.load(f)
    print(f"client sizes: {[len(c) for c in CLIENT_INDICES]}")

## 🗂️ Dataset (exact Part-2 pipeline: samples + cache)

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- normal augmentation for originals (Part-1 aug_normal equivalent) ---
aug_train = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.1, 0.1), rotate=(-30, 30), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.RandomGamma(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)

# --- strong augmentation for oversampled copies (Part-1 aug_strong equivalent) ---
aug_strong = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(translate_percent=0.1, scale=(0.8, 1.2), rotate=(-45, 45), p=0.7),
    A.OneOf([A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.5),
             A.GridDistortion(p=0.5)], p=0.4),
    A.OneOf([A.RandomBrightnessContrast(p=0.5),
             A.HueSaturationValue(p=0.5),
             A.CLAHE(p=0.5)], p=0.5),
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)

aug_eval = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)


class FloodSegDataset(Dataset):
    """Original images use aug_normal; oversampled copies use aug_strong.
    flags is None for val/test (deterministic aug_eval)."""
    def __init__(self, samples, transform, transform_strong=None, flags=None):
        self.samples = samples
        self.transform = transform
        self.transform_strong = transform_strong
        self.flags = flags

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img  = cv2.imread(s["img"]);           img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE)
        if img.shape[:2] != mask.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        if self.flags is not None and self.transform_strong is not None and self.flags[idx]:
            transform = self.transform_strong
        else:
            transform = self.transform
        aug = transform(image=img, mask=mask)
        return aug["image"], aug["mask"].long(), os.path.basename(s["img"])


# --- load (oversampled) train samples + copy flags from Part 1 ---
_ov_path = os.path.join(PART1_DIR, "train_samples_oversampled.json")
_fl_path = os.path.join(PART1_DIR, "oversample_flags.json")
if os.path.exists(_ov_path) and os.path.exists(_fl_path):
    with open(_ov_path) as f:
        train_samples = json.load(f)
    with open(_fl_path) as f:
        oversample_flags = json.load(f)
    print(f"Loaded oversampled train set ({len(train_samples)} samples, "
          f"{int(sum(oversample_flags))} copies).")
else:
    with open(os.path.join(PART1_DIR, "train_samples.json")) as f:
        train_samples = json.load(f)
    oversample_flags = [False] * len(train_samples)
    print("Oversampled files not found -> using originals only (no copies).")

with open(os.path.join(PART1_DIR, "val_samples.json")) as f:
    val_samples = json.load(f)
with open(os.path.join(PART1_DIR, "test_samples.json")) as f:
    test_samples = json.load(f)

# --- remap Part-1 absolute paths to THIS session's dataset mounts ---
# Part 1 recorded full paths (e.g. under /kaggle/input/datasets/...). The same
# dataset can be mounted under a different slug in the Part-2 notebook, so we
# re-point img/mask paths at the roots that actually exist here.
def _find_dataset_roots():
    roots = {}
    fb = os.environ.get("FLOODNET_BASE", "")
    sb = os.environ.get("SECONDARY_BASE", "")
    if fb and os.path.isdir(fb):
        roots["floodnet"] = fb
    if sb and os.path.isdir(sb):
        roots["secondary"] = sb
    if IN_KAGGLE and (not roots.get("floodnet") or not roots.get("secondary")):
        for d in os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else []:
            base = os.path.join("/kaggle/input", d)
            if "floodnet" not in roots:
                cand = os.path.join(base, "FloodNet Challenge - Track 1")
                if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "Train", "Labeled")):
                    roots["floodnet"] = cand
            if "floodnet" not in roots:
                for sub in os.listdir(base) if os.path.isdir(base) else []:
                    cand = os.path.join(base, sub, "FloodNet Challenge - Track 1")
                    if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "Train", "Labeled")):
                        roots["floodnet"] = cand
            if "secondary" not in roots:
                cand = os.path.join(base, "floodnet_fixed")
                if os.path.isdir(os.path.join(cand, "data_finally")):
                    roots["secondary"] = cand
            if "secondary" not in roots:
                for sub in os.listdir(base) if os.path.isdir(base) else []:
                    cand = os.path.join(base, sub, "floodnet_fixed")
                    if os.path.isdir(os.path.join(cand, "data_finally")):
                        roots["secondary"] = cand
    return roots

def _remap_path(p, roots):
    if os.path.exists(p):
        return p
    for marker, key in [("/data_finally/", "secondary"),
                        ("/Train/Labeled/", "floodnet"),
                        ("FloodNet Challenge - Track 1", "floodnet")]:
        i = p.find(marker)
        if i != -1 and roots.get(key):
            cand = os.path.join(roots[key], p[i + len(marker):].lstrip("/"))
            if os.path.exists(cand):
                return cand
    return p

_ds_roots = _find_dataset_roots()
_n_remap = 0
for _samples in (train_samples, val_samples, test_samples):
    for _s in _samples:
        for _k in ("img", "mask"):
            _new = _remap_path(_s[_k], _ds_roots)
            if _new != _s[_k]:
                _s[_k] = _new
                _n_remap += 1
print(f"Remapped {_n_remap} Part-1 sample paths to this session's dataset mount "
      f"(floodnet={_ds_roots.get('floodnet')} secondary={_ds_roots.get('secondary')})")

# ---- FAST-320 v3: pre-resize cache (kills the full-res CPU decode floor) ----
# Per-epoch time was FLAT across IMG_SIZE (320/384/512 ~ same) -> the wall was
# cv2.imread at full original resolution + resize per __getitem__, not GPU pixel
# work. Resize every image once now; loaders then read small cached files.
try:
    if os.environ.get("FSCG_CACHE", "1") == "1":
        _cache_dir = os.path.join(WORKING_DIR, "resized_cache", str(IMG_SIZE))
        os.makedirs(_cache_dir, exist_ok=True)

        def _resize_to_cache(s, split):
            img = cv2.imread(s["img"])
            msk = cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE)
            if img is None or msk is None:
                return s
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
            msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
            stem = os.path.splitext(os.path.basename(s["img"]))[0]
            _ip = os.path.join(_cache_dir, f"{split}_{stem}.jpg")
            _mp = os.path.join(_cache_dir, f"{split}_{stem}_mask.png")
            cv2.imwrite(_ip, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            cv2.imwrite(_mp, msk)
            return {"img": _ip, "mask": _mp}

        _t0c = time.time()
        import concurrent.futures as _cf
        _tasks = ([(s, "tr") for s in train_samples] +
                  [(s, "va") for s in val_samples] +
                  [(s, "te") for s in test_samples])
        _nw = max(1, min(4, os.cpu_count() or 1))
        with _cf.ThreadPoolExecutor(max_workers=_nw) as _ex:
            _out = list(_ex.map(_resize_to_cache,
                                [t[0] for t in _tasks], [t[1] for t in _tasks]))
        _ntr = len(train_samples)
        train_samples = _out[:_ntr]
        val_samples   = _out[_ntr:_ntr + len(val_samples)]
        test_samples  = _out[_ntr + len(val_samples):]
        print(f"Pre-resize cache ready: {len(_tasks)} samples -> {_cache_dir} "
              f"({time.time()-_t0c:.0f}s, {_nw} procs); loaders now read small cached files")
    else:
        print("FSCG_CACHE=0 -> reading full-res originals (slow path)")
except Exception as _ec:
    print("Pre-resize cache FAILED -> continuing with full-res originals:", _ec)


train_dataset = FloodSegDataset(train_samples, aug_train, aug_strong, oversample_flags)
val_dataset   = FloodSegDataset(val_samples,   aug_eval)
test_dataset  = FloodSegDataset(test_samples,  aug_eval)
print(f"train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")


## 🚚 Loaders

In [ ]:
_wk = WORKERS if (not SMOKE_TEST and HAS_CUDA) else 0
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=_wk, pin_memory=True,
                          persistent_workers=(_wk > 0), drop_last=True)
val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=_wk, pin_memory=True,
                          persistent_workers=(_wk > 0))
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=_wk, pin_memory=True,
                          persistent_workers=(_wk > 0))
print(f"Loaders ready (workers={_wk}).")

## 📊 Loss (Part-2 composite)

In [ ]:
def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum().float()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1. - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def _mean(l, empty=0.0):
    l = iter(l)
    try:
        acc = next(l)
    except StopIteration:
        return empty
    n = 1
    for v in l:
        acc += v
        n += 1
    return acc / n


def lovasz_softmax_flat(probas, labels, classes="present"):
    """probas: [N, C]; labels: [N]. Sub-sampled, fast multi-class Lovasz."""
    if probas.numel() == 0:
        return probas * 0.0
    C = probas.size(1)
    losses = []
    for c in range(C):
        fg = (labels == c).float()
        if classes == "present" and fg.sum() == 0:
            continue
        class_pred = probas[:, c]
        errors = (fg - class_pred).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        fg_sorted = fg[perm]
        losses.append(torch.dot(errors_sorted, lovasz_grad(fg_sorted)))
    return _mean(losses)


def lovasz_softmax(pred, target, classes="present", max_pixels=65536):
    """Sub-sample pixels to keep Lovasz fast on high-res masks."""
    probas = F.softmax(pred, dim=1)
    B, C, H, W = probas.shape
    n_pix = H * W
    if n_pix > max_pixels:
        idx = torch.randperm(n_pix, device=probas.device)[:max_pixels]
        probas = probas.reshape(B, C, -1)[:, :, idx].permute(0, 2, 1).reshape(-1, C)
        target = target.reshape(B, -1)[:, idx].reshape(-1)
    else:
        probas = probas.reshape(-1, C)
        target = target.reshape(-1)
    return lovasz_softmax_flat(probas, target, classes=classes)


class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, pred, target):
        pred = torch.softmax(pred, dim=1)
        onehot = torch.nn.functional.one_hot(target, self.num_classes).permute(0, 3, 1, 2).float()
        inter = (pred * onehot).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + onehot.sum(dim=(2, 3))
        dice = (2 * inter + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=3.0, weight=None, ignore_index=-100):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, pred, target):
        ce = F.cross_entropy(pred, target, weight=self.weight,
                             reduction="none", ignore_index=self.ignore_index)
        pt = torch.exp(-ce)
        return (self.alpha * (1 - pt) ** self.gamma * ce).mean()


class AdvancedLoss(nn.Module):
    def __init__(self, num_classes=10, class_weights=None, device=DEVICE):
        super().__init__()
        if class_weights is not None:
            class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = DiceLoss(num_classes)
        self.focal = FocalLoss(alpha=0.75, gamma=3.0, weight=class_weights)

    def forward(self, main_out, aux2, aux3, target):
        main_ce = self.ce(main_out, target)
        main_dice = self.dice(main_out, target)
        main_focal = self.focal(main_out, target)
        main_lovasz = lovasz_softmax(main_out, target)
        loss_main = 0.4 * main_ce + 0.35 * main_dice + 0.15 * main_focal + 0.4 * main_lovasz
        aux2_loss = 0.15 * (self.ce(aux2, target) + self.dice(aux2, target))
        aux3_loss = 0.10 * (self.ce(aux3, target) + self.dice(aux3, target))
        return loss_main + aux2_loss + aux3_loss


def val_loss_fn(pred, target, criterion):
    return 0.5 * criterion.ce(pred, target) + 0.5 * criterion.dice(pred, target)

## 📊 Metrics + TTA

In [ ]:

def _logits(model, x):
    y = model(x)
    return y[0] if isinstance(y, (tuple, list)) else y
@torch.no_grad()
def compute_miou(model, loader, num_classes=NUM_CLASSES, use_tta=False):
    """GPU confusion-matrix mIoU (Optimization 2) + pixel accuracy + per-class IoU."""
    model.eval()
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64, device=DEVICE)
    pix_correct = 0
    pix_total = 0
    for imgs, msks, _ in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        msks = msks.to(DEVICE, non_blocking=True)
        if use_tta:
            preds = tta_predict(model, imgs).argmax(dim=1)
        else:
            preds = torch.argmax(_logits(model, imgs), dim=1)
        valid = (msks >= 0) & (msks < num_classes)
        gts = msks[valid]
        prs = preds[valid]
        cm += torch.bincount(num_classes * gts + prs, minlength=num_classes ** 2).reshape(
            num_classes, num_classes)
        pix_correct += (preds == msks).sum().item()
        pix_total += msks.numel()
    inter = torch.diag(cm).float()
    union = cm.sum(dim=1).float() + cm.sum(dim=0).float() - inter
    per_class = torch.where(union > 0, inter / torch.clamp(union, min=1e-8),
                            torch.zeros_like(inter))
    valid = union > 0
    miou = per_class[valid].mean().item() if valid.any() else 0.0
    return miou, {c: float(per_class[c].item()) for c in range(num_classes)}, \
           pix_correct / max(1, pix_total)


def compute_per_class_stats(preds, targets, num_classes):
    preds = preds.argmax(dim=1)
    stats = []
    for cls in range(num_classes):
        p = (preds == cls)
        t = (targets == cls)
        stats.append([(p & t).sum().item(), p.sum().item(), t.sum().item()])
    return stats


def dice_score(pred, target, num_classes=10, smooth=1e-6):
    pred = torch.argmax(pred, dim=1)
    dice = 0.0
    for cls in range(num_classes):
        pc = (pred == cls).float()
        tc = (target == cls).float()
        dice += (2 * (pc * tc).sum() + smooth) / (pc.sum() + tc.sum() + smooth)
    return dice / num_classes


def iou_score(pred, target, num_classes=10, smooth=1e-6):
    pred = torch.argmax(pred, dim=1)
    iou = 0.0
    for cls in range(num_classes):
        pc = (pred == cls)
        tc = (target == cls)
        ic = (pc & tc).float().sum()
        uc = (pc | tc).float().sum()
        iou += (ic + smooth) / (uc + smooth)
    return iou / num_classes


def pixel_accuracy(pred, target):
    pred = torch.argmax(pred, dim=1)
    return (pred == target).float().mean()


@torch.no_grad()
def tta_predict(model, images):
    """Test-time augmentation: original + hflip + vflip + both."""
    preds = []
    main = _logits(model, images)
    preds.append(main)
    h = torch.flip(images, dims=[3]); main_h = _logits(model, h)
    preds.append(torch.flip(main_h, dims=[3]))
    v = torch.flip(images, dims=[2]); main_v = _logits(model, v)
    preds.append(torch.flip(main_v, dims=[2]))
    hv = torch.flip(images, dims=[2, 3]); main_hv = _logits(model, hv)
    preds.append(torch.flip(main_hv, dims=[2, 3]))
    return torch.mean(torch.stack([torch.softmax(p, dim=1) for p in preds]), dim=0)

## 🧠 Model architecture

In [ ]:
def _make_model():
    import segmentation_models_pytorch as smp
    return smp.Unet(encoder_name='resnet34', encoder_weights=(None if SMOKE_TEST else 'imagenet'),
                    in_channels=3, classes=NUM_CLASSES)

## 🏋️ Train (identical recipe to Part-2 FSCG-UNet)

In [ ]:
class MainLoss(nn.Module):
    """Main term of the Part-2 composite loss (CE + Dice + Focal + Lovasz),
    model-agnostic (takes plain logits + target)."""
    def __init__(self, num_classes, class_weights=None):
        super().__init__()
        if class_weights is not None:
            class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
        self.ce    = nn.CrossEntropyLoss(weight=class_weights)
        self.dice  = DiceLoss(num_classes)
        self.focal = FocalLoss(alpha=0.75, gamma=3.0, weight=class_weights)

    def forward(self, logits, target):
        return (0.4 * self.ce(logits, target)
                + 0.35 * self.dice(logits, target)
                + 0.15 * self.focal(logits, target)
                + 0.4 * lovasz_softmax(logits, target))
print("MainLoss ready (same composite as Part-2 FSCG-UNet).")

In [ ]:
def train_centralized(name, model, epochs=EPOCHS, lr=3e-5):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7)
    criterion = MainLoss(NUM_CLASSES, class_weights).to(DEVICE)
    scaler = torch.amp.GradScaler("cuda", enabled=HAS_CUDA) if HAS_CUDA else None
    history = {"epoch": [], "train_loss": [], "val_miou": []}
    best_iou, best_state = -1.0, None
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        if (time.time() - t0) / 3600 >= MAX_HOURS:
            print(f"  [TIME-BUDGET] {MAX_HOURS:.1f}h reached after {epoch-1} epochs; stopping (best checkpoint kept).")
            break
        model.train()
        total_loss, n_seen = 0.0, 0
        loop = tqdm(train_loader, desc=f"[{name}] epoch {epoch}")
        for imgs, msks, _ in loop:
            imgs = imgs.to(DEVICE, non_blocking=True)
            msks = msks.long().to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=HAS_CUDA):
                logits = model(imgs)
                if isinstance(logits, (tuple, list)):
                    logits = logits[0]
                loss = criterion(logits, msks)
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                scaler.step(optimizer); scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                optimizer.step()
            total_loss += loss.item() * imgs.size(0); n_seen += imgs.size(0)
            loop.set_postfix(loss=loss.item())
        scheduler.step()
        vm, per_class, _ = compute_miou(model, val_loader, use_tta=True)  # real val GT, deterministic
        history["epoch"].append(epoch)
        history["train_loss"].append(total_loss / max(1, n_seen))
        history["val_miou"].append(vm)
        if vm > best_iou:
            best_iou = vm
            best_state = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), os.path.join(CKPT_DIR, "best_val_iou.pth"))
        print(f"[{name}] epoch {epoch:3d}/{epochs}  loss={history['train_loss'][-1]:.4f}  "
              f"val_mIoU={vm:.4f}  best={best_iou:.4f}")
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_iou


def _sanity(model, tag):
    model.eval()
    with torch.no_grad():
        y = model(torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE))
        if isinstance(y, (tuple, list)):
            y = y[0]
        print(f"{tag}: out={tuple(y.shape)} params={sum(p.numel() for p in model.parameters())/1e6:.2f}M")
        assert y.shape[-2:] == (IMG_SIZE, IMG_SIZE), "model output resolution must equal IMG_SIZE"


print("Training functions ready.")
model = _make_model().to(DEVICE)
_sanity(model, "UNet (ResNet-34)")

In [ ]:
t0 = time.time()
model, history, best_val = train_centralized('UNet (ResNet-34)', model)
print(f'Training done in {(time.time()-t0)/60:.1f} min | best val mIoU = {best_val:.4f}')

## 🎯 Final evaluation on held-out REAL test (TTA)

In [ ]:
model.load_state_dict(torch.load(os.path.join(CKPT_DIR, "best_val_iou.pth"),
                                 map_location=DEVICE))
test_miou, test_per_class, test_pix = compute_miou(model, test_loader, use_tta=True)
print(f"\n===== UNet (ResNet-34) -- TEST (TTA, real GT) =====")
print(f"  mIoU      : {test_miou:.4f}")
print(f"  Pixel Acc : {test_pix:.4f}")

result = {
    "model_id": "unet_resnet34",
    "display_name": "UNet (ResNet-34)",
    "type": "centralized (literature)",
    "params_m": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
    "epochs_trained": len(history["epoch"]),
    "heldout_miou": test_miou,
    "pixel_acc": test_pix,
    "best_val_miou": best_val,
    "per_class_iou": {str(k): v for k, v in test_per_class.items()},
    "history": history,
    "config": {"img_size": IMG_SIZE, "batch": BATCH_SIZE, "lr": 3e-5,
               "epochs": EPOCHS, "n_train": len(train_dataset)},
    "source_ref": "[36] Karc\u0131 et al., *Natural Hazards*, 2026 -- UNet+ResNet comparison",
}
with open(os.path.join(OUTPUT_DIR, "result_unet_resnet34.json"), "w") as f:
    json.dump(result, f, indent=2, default=float)
print(f"Saved: result_unet_resnet34.json")

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(history["epoch"], history["train_loss"], "o-", color="tab:red", label="train loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Train loss", color="tab:red")
ax2 = ax1.twinx()
ax2.plot(history["epoch"], history["val_miou"], "s-", color="tab:blue", label="val mIoU (real GT)")
ax2.set_ylabel("val mIoU", color="tab:blue")
plt.title("UNet (ResNet-34) -- Training Curve")
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "31_curve_unet_resnet34.png"), dpi=150, bbox_inches="tight")
plt.show(); print("31_curve_unet_resnet34.png saved.")

## 📦 Package outputs

In [ ]:
import shutil, datetime, zipfile
_t = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
_zip = os.path.join(WORKING_DIR, f"Part3_unet_resnet34_{_t}.zip")
with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for _r, _, _fs in os.walk(OUTPUT_DIR):
        for _f in _fs:
            if _f.endswith(".pth"):
                continue
            _p = os.path.join(_r, _f)
            z.write(_p, os.path.relpath(_p, WORKING_DIR))
    z.write(os.path.join(CKPT_DIR, "best_val_iou.pth"), "checkpoints/best_val_iou.pth")
print(f"Done: {_zip} ({os.path.getsize(_zip)/1e6:.2f} MB)")
if IN_KAGGLE:
    print("Kaggle: /kaggle/working/ is auto-saved. Download from the Output tab.")